In [1]:
import tensorflow as tf
from tensorflow.keras.layers import Dense, Conv2D, Reshape, LayerNormalization, Embedding
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.text import Tokenizer
import numpy
import time
import os
print("All modules imported successfully!")

I0000 00:00:1779948791.291160    1631 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1779948791.796418    1631 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779948793.741169    1631 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


All modules imported successfully!


In [2]:
class Encoder(tf.keras.layers.Layer):
    def __init__(self, emb, latent, heads=6, **kwargs):
        super().__init__(**kwargs)
        self.emb = emb
        self.latent = latent
        self.heads = heads

        # 1. Separate projections for Q, K, and V for all heads combined
        self.wq = Dense(self.latent * self.heads)
        self.wk = Dense(self.latent * self.heads)
        self.wv = Dense(self.latent * self.heads)
        
        # 2. Final projection to ensure the output matches 'emb' size for the residual connection
        self.dense_proj = Dense(self.emb)

        self.dense = Sequential([
            Dense(latent, activation='relu'),
            Dense(emb)
        ])
        
        # Initialize Layer Normalization blocks
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)

    def pos_enc(self, duncan_seq_len):
        pos = tf.cast(tf.range(duncan_seq_len), tf.float32)[:, tf.newaxis]
        dimensions = tf.cast(tf.range(self.emb), tf.float32)[tf.newaxis, :]
        angle_rates = 1 / tf.pow(10000.0, (2 * (dimensions // 2)) / tf.cast(self.emb, tf.float32))
        angle_rads = pos * angle_rates
        
        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])
        
        pos_encoding = tf.stack([sines, cosines], axis=-1)
        pos_encoding = tf.reshape(pos_encoding, [duncan_seq_len, self.emb])
        return pos_encoding

    def split_heads(self, x, batch_size):
        """
        Splits the last dimension into (heads, latent).
        Transposes the result so the shape is (batch_size, heads, seq_len, latent)
        """
        x = tf.reshape(x, (batch_size, -1, self.heads, self.latent))
        return tf.transpose(x, perm=[0, 2, 1, 3])

    def call(self, inputs, mask=None):
        """
        inputs: (batch_size, duncan_seq_len, emb)
        """
        bs = tf.shape(inputs)[0] # Dynamic batch size
        sl = tf.shape(inputs)[1] # Dynamic sequence length
        
        x = inputs
        # 1. Add Positional Encoding
        # pos_enc = self.pos_enc(sl) 
        # x = inputs + pos_enc # (batch_size, seq_len, emb)

        # 2. Linear Projections
        q = self.wq(x)  # (batch_size, seq_len, latent * heads)
        k = self.wk(x)  # (batch_size, seq_len, latent * heads)
        v = self.wv(x)  # (batch_size, seq_len, latent * heads)

        # 3. Split into Multi-Heads
        q = self.split_heads(q, bs) # (batch_size, heads, seq_len, latent)
        k = self.split_heads(k, bs) # (batch_size, heads, seq_len, latent)
        v = self.split_heads(v, bs) # (batch_size, heads, seq_len, latent)

        # 4. Scaled Dot-Product Attention
        # transpose_b=True swaps the last two dimensions of K -> (batch_size, heads, latent, seq_len)
        attention_scores = tf.matmul(q, k, transpose_b=True) # (batch_size, heads, seq_len, seq_len)
        attention_scores = attention_scores / tf.math.sqrt(tf.cast(self.latent, tf.float32))

        if mask is not None:
            # Multiply the mask by a huge negative number and add it to the scores.
            # Assuming duncan_mask has 1s for pads and 0s for real tokens.
            attention_scores += (mask * -1e9)
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)
        
        # Multiply weights by V
        attention_output = tf.matmul(attention_weights, v) # (batch_size, heads, seq_len, latent)

        # 5. Recombine Heads
        # Transpose back to (batch_size, seq_len, heads, latent)
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        # Flatten the last two dimensions
        concat_attention = tf.reshape(attention_output, (bs, sl, self.latent * self.heads)) # (batch_size, seq_len, latent * heads)

        # 6. Final Linear Projection and First Residual
        proj_attention = self.dense_proj(concat_attention) # (batch_size, seq_len, emb)
        out1 = self.layernorm1(proj_attention + x) # Residual connection now works perfectly

        # 7. Feed-Forward Network and Second Residual
        ffn_output = self.dense(out1) # (batch_size, seq_len, emb)
        encoder_output = self.layernorm2(ffn_output + out1) # (batch_size, seq_len, emb)

        return encoder_output

In [3]:
class Decoder(tf.keras.layers.Layer):
    def __init__(self, emb, latent, heads=6, **kwargs):
        super().__init__(**kwargs)
        self.emb = emb
        self.latent = latent
        self.heads = heads

        # 1. Weights for Masked Self-Attention
        self.wq_self = Dense(self.latent * self.heads)
        self.wk_self = Dense(self.latent * self.heads)
        self.wv_self = Dense(self.latent * self.heads)
        self.dense_proj_self = Dense(self.emb)

        # 2. Weights for Cross-Attention (Encoder-Decoder)
        self.wq_cross = Dense(self.latent * self.heads)
        self.wk_cross = Dense(self.latent * self.heads)
        self.wv_cross = Dense(self.latent * self.heads)
        self.dense_proj_cross = Dense(self.emb)

        # 3. Feed-Forward Network
        self.dense = Sequential([
            Dense(latent, activation='relu'),
            Dense(emb)
        ])
        
        self.layernorm1 = LayerNormalization(epsilon=1e-6)
        self.layernorm2 = LayerNormalization(epsilon=1e-6)
        self.layernorm3 = LayerNormalization(epsilon=1e-6)

    def split_heads(self, x, batch_size):
        x = tf.reshape(x, (batch_size, -1, self.heads, self.latent))
        return tf.transpose(x, perm=[0, 2, 1, 3])
    
    def pos_enc(self, duncan_seq_len, emb):
        pos = tf.cast(tf.range(duncan_seq_len), tf.float32)[:, tf.newaxis]
        dimensions = tf.cast(tf.range(emb), tf.float32)[tf.newaxis, :]
        angle_rates = 1 / tf.pow(10000.0, (2 * (dimensions // 2)) / tf.cast(emb, tf.float32))
        angle_rads = pos * angle_rates
        
        sines = tf.math.sin(angle_rads[:, 0::2])
        cosines = tf.math.cos(angle_rads[:, 1::2])
        
        pos_encoding = tf.stack([sines, cosines], axis=-1)
        pos_encoding = tf.reshape(pos_encoding, [duncan_seq_len, emb])
        return pos_encoding
    
    def call(self, inputs, enc_output, look_ahead_mask=None, duncan_padding_mask=None):
        """
        inputs: (batch_size, tar_seq_len, emb)
        enc_output: (batch_size, inp_seq_len, emb)
        """
        # Using tf.shape()[index] is safer for dynamic batch sizing in graph mode
        bs = tf.shape(inputs)[0] 
        sl = tf.shape(inputs)[1] 

        # --- BLOCK 1: MASKED SELF-ATTENTION ---
        pos_enc = self.pos_enc(sl, self.emb)
        x = inputs + pos_enc

        q = self.wq_self(x) 
        k = self.wk_self(x) 
        v = self.wv_self(x) 

        q = self.split_heads(q, bs) 
        k = self.split_heads(k, bs) 
        v = self.split_heads(v, bs) 

        attention_scores = tf.matmul(q, k, transpose_b=True) 
        attention_scores = attention_scores / tf.math.sqrt(tf.cast(self.latent, tf.float32))

        # Apply the Look-Ahead Mask (Autoregressive feature)
        if look_ahead_mask is not None:
            attention_scores += (look_ahead_mask * -1e9)
            
        attention_weights = tf.nn.softmax(attention_scores, axis=-1)
        attention_output = tf.matmul(attention_weights, v) 
        attention_output = tf.transpose(attention_output, perm=[0, 2, 1, 3])
        concat_attention = tf.reshape(attention_output, (bs, sl, self.latent * self.heads)) 
        
        proj_attention = self.dense_proj_self(concat_attention) 
        proj_attention = self.layernorm1(proj_attention + x) 

        # --- BLOCK 2: CROSS-ATTENTION ---
        # Keys and Values come from the Encoder's output
        enc_k = self.wk_cross(enc_output)  
        enc_v = self.wv_cross(enc_output)  
        
        # Query comes from the previous Decoder block
        dec_q = self.wq_cross(proj_attention)  
        
        dec_q = self.split_heads(dec_q, bs) 
        enc_k = self.split_heads(enc_k, bs) 
        enc_v = self.split_heads(enc_v, bs) 

        cross_attention_scores = tf.matmul(dec_q, enc_k, transpose_b=True) 
        cross_attention_scores = cross_attention_scores / tf.math.sqrt(tf.cast(self.latent, tf.float32))
        
        # Apply the Padding Mask (Ignore padded encoder tokens)
        if duncan_padding_mask is not None:
            cross_attention_scores += (duncan_padding_mask * -1e9)

        cross_attention_weights = tf.nn.softmax(cross_attention_scores, axis=-1)
        cross_attention_output = tf.matmul(cross_attention_weights, enc_v) 
        cross_attention_output = tf.transpose(cross_attention_output, perm=[0, 2, 1, 3])
        concat_cross_attention = tf.reshape(cross_attention_output, (bs, sl, self.latent * self.heads)) 
        
        proj_cross_attention = self.dense_proj_cross(concat_cross_attention) 
        proj_cross_attention = self.layernorm2(proj_cross_attention + proj_attention) 

        # --- BLOCK 3: FEED-FORWARD NETWORK ---
        ffn_output = self.dense(proj_cross_attention) 
        decoder_output = self.layernorm3(ffn_output + proj_cross_attention) 
        
        return decoder_output

In [4]:
class PatchEmbedding(tf.keras.layers.Layer):
    def __init__(self, image_size=256, patch_size=16, emb_dim=512, **kwargs):
        super().__init__(**kwargs)
        self.image_size = image_size
        self.patch_size = patch_size
        self.emb_dim = emb_dim
        self.num_patches = (image_size // patch_size) ** 2

        self.projection = Conv2D(filters=emb_dim, kernel_size=patch_size,strides=patch_size, padding='valid')

        self.flatten_sequence = Reshape((self.num_patches, self.emb_dim))
        self.position_emb = tf.Variable(
            initial_value=tf.random.normal([1, self.num_patches, self.emb_dim], stddev=0.02), trainable=True
        )

    def call(self, images):
        patches = self.projection(images)
        seq = self.flatten_sequence(patches)
        encoded_seq = seq + self.position_emb
        return encoded_seq

In [5]:
class ViT_Captioning_Transformer(tf.keras.Model):
    def __init__(self, num_layers, emb, latent, heads, target_vocab_size, **kwargs):
        super().__init__(**kwargs)
        
        # The new Vision front-end
        self.image_patcher = PatchEmbedding(image_size=256, patch_size=16, emb_dim=emb)
        
        self.text_embedding = Embedding(target_vocab_size, emb) #it is fix
        # Your custom blocks (Make sure Encoder no longer adds pos_enc)
        self.encoder = [Encoder(emb, latent, heads) for _ in range(num_layers)]
        self.decoder = [Decoder(emb, latent, heads) for _ in range(num_layers)]
        
        self.final_layer = Dense(target_vocab_size)

    def call(self, image_inputs, targets, training=True, look_ahead_mask=None, dec_padding_mask=None):
        
        # 1. Turn the image into a sequence of positional patches
        # Output shape: (batch_size, 256, emb)
        enc_output = self.image_patcher(image_inputs)
        
        # 2. Pass through Encoders
        for enc in self.encoder:
            # Note: We don't need an enc_padding_mask here because image patches 
            # are perfectly square and never padded with zeros!
            enc_output = enc(enc_output, mask=None) 
            
        # 3. Pass through Decoders
        # dec_output = targets
        dec_output = self.text_embedding(targets)
        for dec in self.decoder:
            dec_output = dec(
                dec_output, 
                enc_output, 
                look_ahead_mask=look_ahead_mask, 
                duncan_padding_mask=None
            )
            
        final_output = self.final_layer(dec_output)
        return final_output

In [ ]:
BATCH_SIZE = 8
BUFFER_SIZE = 1000
EMB_DIM = 256
UNITS = 512
VOCAB_SIZE = 5000
MAX_LEN = 40

image_folder = "/mnt/f/Abhinav/Mystic/Machine Learning/DeepLearning/Images"
caption_file = "/mnt/f/Abhinav/Mystic/Machine Learning/DeepLearning/captions.txt"

image_paths = []
captions = []

with open(caption_file, 'r') as f:
    next(f)
    for line in f:
        img_name, caption = line.strip().split(',', 1)
        image_paths.append(os.path.join(image_folder, img_name))
        captions.append(f"<start> {caption} <end>")
tokenizer = Tokenizer(num_words=VOCAB_SIZE,oov_token="<unk>", filters="!\"#$%&()*+.,-/:;=?@[\]^_`{|}~")
tokenizer.fit_on_texts(captions)
tokenizer.word_index['<pad>'] = 0
tokenizer.index_word[0] = '<pad>'

train_seq = tokenizer.texts_to_sequences(captions)
cap_vector = tf.keras.preprocessing.sequence.pad_sequences(train_seq, maxlen=MAX_LEN, padding='post')

def load_image(image_path, cap):
    img = tf.io.read_file(image_path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (256, 256))
    img = img / 255.0
    return img, cap

dataset = tf.data.Dataset.from_tensor_slices((image_paths, cap_vector))
dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("\n--- Data Pipeline Sanity Check ---")
for img_batch, cap_batch in dataset.take(1):
    print(f"Image batch shape: {img_batch.shape} -> (Batch Size, Height, Width, Channels)")
    print(f"Caption batch shape: {cap_batch.shape} -> (Batch Size, Max Sequence Length)")
print("----------------------------------\n")

I0000 00:00:1779948797.951446    1631 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3537 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9



--- Data Pipeline Sanity Check ---
Image batch shape: (64, 256, 256, 3) -> (Batch Size, Height, Width, Channels)
Caption batch shape: (64, 40) -> (Batch Size, Max Sequence Length)
----------------------------------



In [7]:
def create_padding_mask(seq):
    """
        seq: (batch_size, seq_len)
    """
    seq = tf.cast(tf.math.equal(seq, 0), tf.float32)
    return seq[:, tf.newaxis, tf.newaxis, :]  # (batch_size, 1, 1, seq_len)

def create_look_ahead_mask(size):
    mask = 1 - tf.linalg.band_part(tf.ones((size, size)), -1, 0)
    return mask  # (seq_len, seq_len)


In [8]:
transformer = ViT_Captioning_Transformer(
    num_layers=4, emb=512, latent=UNITS, heads=8, target_vocab_size=VOCAB_SIZE
)
optimizer = tf.keras.optimizers.Adam()

loss_obj = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True,reduction='none')

def loss_function(real, pred):
    mask = tf.math.logical_not(tf.math.equal(real, 0))
    loss_ = loss_obj(real, pred)
    mask = tf.cast(mask, dtype=loss_.dtype)
    loss_ *= mask
    return tf.reduce_sum(loss_) / tf.reduce_sum(mask)

print("\n--- Initializing Model & Counting Parameters ---")

# Pull one batch to build the model's weights
for dummy_img, dummy_cap in dataset.take(1):
    dummy_dec_input = dummy_cap[:, :-1]
    
    # Create quick dummy masks
    dummy_look_ahead = create_look_ahead_mask(tf.shape(dummy_dec_input)[1])
    dummy_pad = create_padding_mask(dummy_dec_input)
    dummy_combined = tf.maximum(dummy_pad, dummy_look_ahead)
    
    # Run a dummy forward pass (training=False so we don't apply dropout)
    _ = transformer(
        image_inputs=dummy_img,
        targets=dummy_dec_input,
        training=False,
        look_ahead_mask=dummy_combined,
        dec_padding_mask=dummy_pad
    )
    break # We only need one pass!

# Now that the weights are built, Keras can print the beautiful summary table
transformer.summary()
print("------------------------------------------------\n")

# training step
@tf.function
def train_step(image_tensor, target_caption):
    dec_input = target_caption[:, :-1]
    real_target = target_caption[:, 1:]

    look_ahead_mask = create_look_ahead_mask(tf.shape(dec_input)[1])
    dec_target_padding_mask = create_padding_mask(dec_input)

    combined_mask = tf.maximum(dec_target_padding_mask, look_ahead_mask)

    with tf.GradientTape() as tape:
        predictions = transformer(
            image_inputs=image_tensor,
            targets=dec_input,
            training=True,
            look_ahead_mask=combined_mask,
            dec_padding_mask=dec_target_padding_mask
        )

        loss = loss_function(real_target, predictions)

    gradients = tape.gradient(loss, transformer.trainable_variables)
    optimizer.apply_gradients(zip(gradients, transformer.trainable_variables))
    return loss



--- Initializing Model & Counting Parameters ---


I0000 00:00:1779948804.198891    1631 cuda_dnn.cc:461] Loaded cuDNN version 91001


Model: "vi_t__captioning__transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ patch_embedding                 │ ?                      │       393,728 │
│ (PatchEmbedding)                │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (64, 39, 512)          │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder (Encoder)               │ ?                      │     8,928,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_1 (Encoder)             │ ?                      │     8,928,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_2 (Encoder)             │ ?                      │     8,928,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ encoder_3 (Encoder)             │ ?                      │     8,928,768 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder (Decoder)               │ ?                      │    17,331,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_1 (Decoder)             │ ?                      │    17,331,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_2 (Decoder)             │ ?                      │    17,331,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_3 (Decoder)             │ ?                      │    17,331,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_64 (Dense)                │ (64, 39, 5000)         │     2,565,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 110,558,600 (421.75 MB)

 Trainable params: 110,558,600 (421.75 MB)

 Non-trainable params: 0 (0.00 B)

------------------------------------------------



In [ ]:
EPOCHS = 30
for epoch in range(EPOCHS):
    start = time.time()
    total_loss = 0
    for batch, (img_tensor, target_caption) in enumerate(dataset):
        batch_loss = train_step(img_tensor, target_caption)
        total_loss += batch_loss

        if batch % 50 == 0:
            print(f"Epoch {epoch+1} Batch {batch} Loss {batch_loss.numpy():.4f}")
    
    print(f"Epoch {epoch+1} Loss {total_loss.numpy()/(batch+1):.4f}")
    print(f"Time taken for 1 epoch: {time.time() - start:.2f} secs\n")
    